# Gemma 4 12B QAT w4a16-ct 테스트

T4에서 동작 확인용. bitsandbytes 없이 compressed-tensors로 로드.

In [ ]:
!pip install -q --upgrade transformers compressed-tensors accelerate
!pip install -q sentence-transformers chromadb

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    print(f"현재 사용: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 1. 모델 로드

In [ ]:
from transformers import AutoModelForMultimodalLM, AutoTokenizer

MODEL_ID = "google/gemma-4-12B-it-qat-w4a16-ct"

print("[1/2] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("[2/2] 모델 로드 (QAT w4a16 compressed-tensors)")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"\n모델 로드 완료 — VRAM: {vram_gb:.2f} GB")

## 2. 간단한 생성 테스트

In [ ]:
messages = [
    {"role": "system", "content": "너는 충남대학교 학내 정보를 안내하는 AI 챗봇이야. 한국어로 답변해."},
    {"role": "user", "content": "충남대학교 컴퓨터융합학부 졸업 요건이 뭐야?"},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

print(f"입력 토큰 수: {inputs['input_ids'].shape[1]}")
print(f"VRAM (입력 후): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.3,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(generated, skip_special_tokens=True)
print(f"\nVRAM (생성 후): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"\n답변:\n{answer}")

## 3. RAG 컨텍스트 포함 테스트 (긴 입력)

In [ ]:
# RAG 시뮬레이션: 컨텍스트가 긴 경우 OOM 여부 확인
fake_context = "충남대학교 컴퓨터융합학부 졸업요건: 총 130학점 이상 이수. " * 50

messages_long = [
    {"role": "system", "content": "너는 충남대학교 학내 정보를 안내하는 AI 챗봇이야. 참고 자료를 기반으로 답변해."},
    {"role": "user", "content": f"참고 자료:\n{fake_context}\n\n질문: 졸업하려면 몇 학점 들어야 해?"},
]

text_long = tokenizer.apply_chat_template(messages_long, tokenize=False, add_generation_prompt=True)
inputs_long = tokenizer(text_long, return_tensors="pt").to(model.device)

print(f"입력 토큰 수: {inputs_long['input_ids'].shape[1]}")

torch.cuda.empty_cache()
with torch.no_grad():
    outputs_long = model.generate(
        **inputs_long,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.3,
    )

generated_long = outputs_long[0][inputs_long["input_ids"].shape[1]:]
answer_long = tokenizer.decode(generated_long, skip_special_tokens=True)
print(f"VRAM (긴 입력 생성 후): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"\n답변:\n{answer_long}")

# 정리
del inputs_long, outputs_long, generated_long
torch.cuda.empty_cache()
print(f"VRAM (정리 후): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## 4. VRAM 요약

In [ ]:
print(f"총 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"현재 할당: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"현재 예약: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"여유: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_reserved()) / 1024**3:.2f} GB")